# DipDECK PCA 34 deduplicated scientific grid

This notebook runs the existing 20-setting, ten-seed, five-fold DipDECK grid on deduplicated phosphatase records in PCA 34. PCA is fitted separately on each training rotation, then applied to its held-out fold. The deduplicated job fits each basis on that rotation's surviving records only. It writes only its own output tree, so it can run alongside the matching record-set notebook.

The upstream CPU tests and CUDA smoke check passed in the two raw-grid notebooks and are intentionally not repeated here. This job still checks the project copy, data archive, patched DipDECK source and assigned CUDA GPU.

Upload this project folder to Google Drive as `MyDrive/explaining-plm-latent-spaces` (or set `PROJECT_DIR`), select **T4 GPU**, and run all cells. The last cell refuses to overwrite an existing completed output tree.


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/explaining-plm-latent-spaces"
CLUSTPY_COMMIT = "82548dc35a01d221474fbb44908b3621d4b97616"
EXPECTED_UPSTREAM_DIPDECK_SHA256 = "c567d24c1722087a75c3eb56f17fe91152b6f818148083d908086b9fc78d438a"
EXPECTED_DIPDECK_SHA256 = "cfbf629ab341bdbb5967af232c24cfc3f2e5fb09cd082fb28b317c45738b7420"
EXPECTED_DATA_ZIP_SHA256 = "3ed7857f71e4c63939ffaee885b485bdb01555d8eced9abc18b567c3711a08e2"
DRIVE_ROOT = PROJECT_DIR
DATA_ZIP = f"{PROJECT_DIR}/data/phosphatase/dipdeck_phosphatase_data.zip"
PCA_DIM = 34
DEDUP = True


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import os
import shutil
import subprocess
import zipfile

repo = Path("/content/project").resolve()
clustpy_source = Path("/content/ClustPy").resolve()
drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)

for target in (repo, clustpy_source):
    if target.parent != Path("/content"):
        raise RuntimeError(f"Refusing to replace unexpected path: {target}")
    if target.exists():
        shutil.rmtree(target)

project_dir = Path(PROJECT_DIR)
if not (project_dir / "src/algo/dipdeck/dipdeck_standalone.py").is_file():
    raise FileNotFoundError(f"Upload this project folder to Google Drive at {project_dir}, then rerun.")
for part in ("src", "data", "results/graphpart", "results/ec_audit"):
    shutil.copytree(project_dir / part, repo / part)
subprocess.run(["git", "clone", "https://github.com/collinleiber/ClustPy.git", str(clustpy_source)], check=True)
subprocess.run(["git", "-C", str(clustpy_source), "checkout", CLUSTPY_COMMIT], check=True)

dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
source_text = dipdeck_source.read_text(encoding="utf-8")
upstream_sha256 = hashlib.sha256(source_text.encode()).hexdigest()
if upstream_sha256 != EXPECTED_UPSTREAM_DIPDECK_SHA256:
    raise RuntimeError(
        f"Unexpected upstream DipDECK source: expected {EXPECTED_UPSTREAM_DIPDECK_SHA256}, "
        f"got {upstream_sha256}"
    )
for old, new in (
    ("self.labels.to(device)", "self.labels = self.labels.to(device)"),
    ("self.cluster_centers.to(device)", "self.cluster_centers = self.cluster_centers.to(device)"),
    ("self.dip_matrix.to(device)", "self.dip_matrix = self.dip_matrix.to(device)"),
):
    if source_text.count(old) != 1:
        raise RuntimeError(f"CUDA patch target is absent or ambiguous: {old}")
    source_text = source_text.replace(old, new)
dipdeck_source.write_text(source_text, encoding="utf-8")
patched_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
if patched_sha256 != EXPECTED_DIPDECK_SHA256:
    raise RuntimeError(
        f"Patched DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {patched_sha256}"
    )

repo_commit = "project-folder"

data_zip = Path(DATA_ZIP)
if not data_zip.is_file():
    raise FileNotFoundError(
        f"Upload dipdeck_phosphatase_data.zip to {data_zip.parent} in Google Drive, then rerun."
    )
data_zip_sha256 = hashlib.sha256(data_zip.read_bytes()).hexdigest()
if data_zip_sha256 != EXPECTED_DATA_ZIP_SHA256:
    raise RuntimeError(
        f"Data ZIP mismatch: expected {EXPECTED_DATA_ZIP_SHA256}, got {data_zip_sha256}"
    )
with zipfile.ZipFile(data_zip) as archive:
    archive.extractall(repo)

required = [
    repo / "data/phosphatase/phosphatase/tmp/prot_t5.h5",
    repo / "data/phosphatase/phosphatase/data_standard.parquetbundle",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Data ZIP has the wrong layout or is incomplete: {missing}")

os.chdir(repo)
print(f"Repository: {repo}")
print(f"Project copied from: {PROJECT_DIR}")
print(f"Data ZIP: {data_zip}")
print(f"Data ZIP SHA256: {data_zip_sha256}")
print(f"ClustPy revision: {CLUSTPY_COMMIT}")

In [ ]:
COLAB_PINS = {
    "Cython": "3.3.0",
    "numpy": "2.1.3",
    "scipy": "1.18.1",
    "scikit-learn": "1.6.1",
    "pandas": "2.2.3",
    "h5py": "3.16.0",
    "pyarrow": "25.0.1",
    "tqdm": "4.70.0",
    "pytest": "9.1.1",
    "psutil": "7.2.2",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
}

import os
import subprocess
import sys

os.chdir("/content")
install_names = [
    "Cython", "numpy", "scipy", "scikit-learn", "pandas",
    "h5py", "pyarrow", "tqdm", "pytest", "psutil",
]
requirements = [f"{name}=={COLAB_PINS[name]}" for name in install_names]
install_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", *requirements],
    check=True, capture_output=True, text=True,
)
print(install_result.stdout)
if install_result.stderr:
    print(install_result.stderr, file=sys.stderr)

if "torch" in sys.modules or "torchvision" in sys.modules:
    raise RuntimeError("torch is already imported. Use Runtime > Restart session, then run all cells.")
torch_result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        f"torch=={COLAB_PINS['torch']}", f"torchvision=={COLAB_PINS['torchvision']}",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ],
    check=True, capture_output=True, text=True,
)
print(torch_result.stdout[-2000:])
if torch_result.stderr:
    print(torch_result.stderr[-2000:], file=sys.stderr)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "--force-reinstall", "--no-cache-dir", "--editable", str(clustpy_source),
] , check=True)

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True,
)
print(pip_check.stdout)
if pip_check.stderr:
    print(pip_check.stderr, file=sys.stderr)
if pip_check.returncode != 0:
    print("pip check reported the Colab base-environment warning above.", file=sys.stderr)

In [ ]:
import hashlib
import importlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
from pathlib import Path

for module_name in (
    "Cython", "h5py", "pandas", "psutil", "pyarrow", "pytest", "scipy", "sklearn", "tqdm",
):
    importlib.import_module(module_name)

import numpy as np
import torch
import torchvision
from threadpoolctl import threadpool_info

if not (clustpy_source / "clustpy/__init__.py").is_file():
    raise FileNotFoundError(f"ClustPy package is missing from the verified checkout: {clustpy_source}")
sys.path.insert(0, str(clustpy_source))
import clustpy
from clustpy.deep import DipDECK
if Path(clustpy.__file__).resolve().parent != (clustpy_source / "clustpy").resolve():
    raise RuntimeError(f"Imported ClustPy is not the verified checkout: {clustpy.__file__}")

observed_versions = {name: metadata.version(name) for name in COLAB_PINS}
version_mismatches = {
    name: {"expected": expected, "observed": observed_versions[name]}
    for name, expected in COLAB_PINS.items()
    if observed_versions[name] != expected
}
if version_mismatches:
    raise RuntimeError(f"Colab dependency mismatch: {version_mismatches}")

dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
dipdeck_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
if dipdeck_sha256 != EXPECTED_DIPDECK_SHA256:
    raise RuntimeError(
        f"DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {dipdeck_sha256}"
    )

environment = {
    "python": platform.python_version(),
    "project_commit": repo_commit,
    "clustpy_version": metadata.version("clustpy"),
    "clustpy_path": clustpy.__file__,
    "clustpy_commit": CLUSTPY_COMMIT,
    "dipdeck_sha256": dipdeck_sha256,
    "package_versions": observed_versions,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_num_threads": torch.get_num_threads(),
    "torch_num_interop_threads": torch.get_num_interop_threads(),
    "thread_environment": {
        name: os.environ.get(name)
        for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    },
    "threadpool_info": threadpool_info(),
}
print(json.dumps(environment, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU, then run all again.")

In [ ]:
standalone = repo / "src/algo/dipdeck/dipdeck_standalone.py"
if not standalone.is_file():
    raise FileNotFoundError(f"Standalone is missing: {standalone}")
output_root = (drive_root / "results/dipdeck").resolve()
owned_directory = output_root / f"scientific_grid_pca{PCA_DIM}{'_dedup' if DEDUP else ''}"
standalone_env = os.environ.copy()
standalone_env["DIPDECK_OUTPUT_ROOT"] = str(output_root)
command = [
    sys.executable, str(standalone), "--run-scientific-pca",
    "--pca-dim", str(PCA_DIM), "--device", "cuda",
    "--clustpy-source-root", str(clustpy_source),
]
validator = [
    sys.executable, str(standalone), "--validate-scientific-pca-output",
    "--pca-dim", str(PCA_DIM), "--output-root", str(output_root),
]
if DEDUP:
    command.append("--dedup")
    validator.append("--dedup")
if owned_directory.exists():
    existing = subprocess.run(validator, cwd=repo)
    if existing.returncode == 0:
        raise FileExistsError(
            f"Output already passed validation: {owned_directory}. Do not overwrite it."
        )
    print(f"Replacing incomplete output: {owned_directory}")
subprocess.run(command, cwd=repo, env=standalone_env, check=True)
subprocess.run(validator, cwd=repo, check=True)
archive_base = drive_root / f"dipdeck_{owned_directory.name}_outputs"
archive_path = archive_base.with_suffix(".zip")
archive_path.unlink(missing_ok=True)
output_archive = Path(shutil.make_archive(
    str(archive_base), "zip", root_dir=output_root, base_dir=owned_directory.name,
))
print(f"Output archive: {output_archive}")
print(f"Output SHA256: {hashlib.sha256(output_archive.read_bytes()).hexdigest()}")
